### **MiniGPT**

In [1]:
import pandas as pd
import torch
import torch.nn as nn

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [26]:
df = pd.read_csv('data/Eng-Fra.csv')
df.head()

,English,French
0,Go.,Va !
1,Go.,Marche.
2,Go.,En route !
3,Go.,Bouge !
4,Hi.,Salut !


In [4]:
from preprocessing.normalize import normalize_text

df['English'] = df['English'].apply(normalize_text)
df['French'] = df['French'].apply(normalize_text)

In [5]:
df.head()

,English,French
0,go,va
1,go,marche
2,go,en route
3,go,bouge
4,hi,salut


In [6]:
sentences = []

for eng in zip(df['English']):
    sentences.append(str(eng))

print(len(sentences))

239189


In [7]:
from preprocessing.vocabulary import Vocabulary

eng_vocab = Vocabulary("eng")

# Step 1: count words
for eng in sentences:
    eng_vocab.add_sentence(eng)
# Step 2: build vocab (top-K)
eng_vocab.build_vocab(max_size=10000)

In [17]:
from utils.GPTDataset import GPTDataset
from models.mini_gpt import MiniGPT
from torch.utils.data import DataLoader

dataset = GPTDataset(sentences, eng_vocab, seq_len=5)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

model = MiniGPT(
    vocab_size=eng_vocab.vocab_size,
    d_model=128,
    heads=4,
    num_layers=2,
    max_len=50
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [21]:
num_epochs = 10

for epoch in range(num_epochs):

    total_loss = 0

    for x, y in dataloader:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        output = model(x)   # [batch, seq_len, vocab]

        output = output.view(-1, output.shape[-1])
        y = y.view(-1)

        loss = criterion(output, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(dataloader):.4f}")

Epoch 1, Loss: 3.3546
Epoch 2, Loss: 3.2508
Epoch 3, Loss: 3.1824
Epoch 4, Loss: 3.1346
Epoch 5, Loss: 3.0967
Epoch 6, Loss: 3.0655
Epoch 7, Loss: 3.0392
Epoch 8, Loss: 3.0167
Epoch 9, Loss: 2.9963
Epoch 10, Loss: 2.9785


In [23]:
def generate(model, start_text, vocab, max_len=10):

    model.eval()

    tokens = [vocab.word2index.get(w, vocab.word2index["UNK"])
              for w in start_text.lower().split()]

    tokens = tokens

    for _ in range(max_len):

        x = torch.tensor(tokens).unsqueeze(0).to(device)

        with torch.no_grad():
            logits = model(x)

        next_token = logits[:, -1, :].argmax(-1).item()

        if next_token == vocab.word2index["EOS"]:
            break

        tokens.append(next_token)

    words = [vocab.index2word[i] for i in tokens]

    return " ".join(words)

In [24]:
print(generate(model, "my name", eng_vocab))

my name on the list',)


In [32]:
torch.save({
    "model_state_dict": model.state_dict(),
    "vocab": eng_vocab.word2index,
    "config": {
        "vocab_size": eng_vocab.vocab_size,
        "d_model": 128,
        "heads": 4,
        "num_layers": 2,
        "max_len": 50
    }
}, "mini_gpt.pth")

In [33]:
from google.colab import files
files.download('mini_gpt.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## **Loading the Model**

In [ ]:
# loading the checkpoint
checkpoint = torch.load("mini_gpt.pth", map_location = device)

# rebuilding vocabulary
eng_vocab = Vocabulary(eng)

eng_vocab.word2index = checkpoint["vocab"]
eng_vocab.index2word = {v:k for k,v in eng_vocab.word2index.items()}
eng_vocab.vocab_size = len(eng_vocab.word2index)

In [11]:
from models.mini_gpt import MiniGPT


config = checkpoint['config']

model = MiniGPT(
    vocab_size = config["vocab_size"],
    d_model = config['d_model'],
    heads = config['heads'],
    num_layers = config['num_layers'],
    max_len = config['max_len']
).to(device)

In [12]:
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

MiniGPT(
  (embedding): Embedding(10004, 128)
  (pos_embedding): Embedding(50, 128)
  (blocks): ModuleList(
    (0-1): 2 x TransformerBlock(
      (attn): MultiHeadAttention(
        (Wq): Linear(in_features=128, out_features=128, bias=True)
        (Wk): Linear(in_features=128, out_features=128, bias=True)
        (Wv): Linear(in_features=128, out_features=128, bias=True)
        (fc_out): Linear(in_features=128, out_features=128, bias=True)
      )
      (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
      (ff): Sequential(
        (0): Linear(in_features=128, out_features=512, bias=True)
        (1): ReLU()
        (2): Linear(in_features=512, out_features=128, bias=True)
      )
      (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    )
  )
  (fc_out): Linear(in_features=128, out_features=10004, bias=True)
)

In [31]:
generate(model, "Tom can", eng_vocab)

'tom can do it if you want to or do that you'